# Burgers equation and shock formation

*fluids · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/fluids/burgers_shock)

The simplest nonlinear fluid equation: fast parts of a wave catch up with slow parts, the profile steepens and, near $t^* = 1/\max(-u_0')$, a shock forms that viscosity keeps thin but smooth. Derivatives are $N\times N$ matrices, so the PDE is one einsum per term, checked against the exact Cole–Hopf solution.

**Vector form**

$$\partial_t u + u\,\partial_x u = \nu\,\partial_x^2 u$$

**Index (tensor) form**

$$\dot u_i = -\tfrac12\,D_{ij}\,u_j^2 + \nu\,L_{ij}\,u_j,\qquad u(x, t) = -2\nu\,\partial_x\ln\varphi,\quad \varphi_t = \nu\,\varphi_{xx}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $u_i$ | `u` | (N,) | velocity at grid point $x_i$ (periodic) |
| $D_{ij}$ | `derivative_matrices(...)[0]` | (N, N) | central difference $(f_{i+1} - f_{i-1})/2\Delta x$ as a matrix |
| $L_{ij}$ | `derivative_matrices(...)[1]` | (N, N) | second difference $(f_{i+1} - 2f_i + f_{i-1})/\Delta x^2$ |
| $D_{ij}\,u_j^2$ | `einsum('ij,j->i', D, 0.5 * u**2)` | (N,) | repeated $j$ ⇒ summed: derivative of the flux $\tfrac12 u^2$ |
| $L_{ij}\,u_j$ | `einsum('ij,j->i', L, u)` | (N,) | viscous term |
| $\textstyle\sum_i u_i\,\Delta x$ | `mass(u, dx)` | (T,) | conserved: the columns of $D$ and $L$ sum to zero |
| $u_\text{CH}(x, t)$ | `cole_hopf(x, t, nu, A, U0)` | (N,) | exact solution, a softmax-weighted average over starting points |

Writing $u\,u_x$ as $(\tfrac12u^2)_x$ (conservation form) makes the scheme conserve $\int u\,dx$ exactly. Without viscosity the characteristics $x = x_0 + u_0(x_0)\,t$ cross at $t^* = 1/\max(-u_0')$ and the solution would become multivalued; viscosity instead forms a shock of width $\sim\nu/\Delta u$. Central differences need $|u|\Delta x/\nu \lesssim 2$, which is why $\nu$ cannot go too low on 512 points. The Cole–Hopf transform $u = -2\nu\,\partial_x\ln\varphi$ turns Burgers into the heat equation, giving the dashed exact curve. A non-zero offset $U_0$ makes the shock travel.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/fluids/burgers_shock/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import mass, slope

N = 512  # grid points


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    nu, A, U0 = params["nu"], params["amplitude"], params["offset"]
    x, dx = tensors.grid(N, dev)  # (N,)
    D, L = tensors.derivative_matrices(N, dx, dev)  # (N, N) each
    u0 = tensors.initial_wave(x, A, U0)  # (N,)

    dt = solve.time_step(dx, abs(U0) + A, nu)
    times, u = solve.simulate(u0, D, L, nu, dt, params["t_max"])  # (T,), (T, N)
    steepness = (-slope(u, D)).max(dim=-1).values  # (T,) steepest downhill slope max(-u_x)

    t_star = 1 / A
    k_form = int((steepness > 0.5 * steepness.max()).int().argmax())
    m = mass(u, dx)
    exact = solve.cole_hopf(x, times[-1].item(), nu, A, U0)
    print(f"{N} points, dt = {dt:.2e}, {len(times) - 1} RK4 steps")
    print(
        f"inviscid breaking time t* = 1/max(-u0') = {t_star:.2f};  "
        f"slope reaches half its peak at t = {times[k_form]:.2f}"
    )
    print(f"mass ∫u dx: start {m[0]:.6f}, end {m[-1]:.6f}")
    print(f"max |u - u_Cole-Hopf| at t = {times[-1]:.2f}: {(u[-1] - exact).abs().max():.2e}")
    return plot.animate(
        x, times, u, lambda t: solve.cole_hopf(x, t, nu, A, U0), steepness, t_star, out_dir / "burgers_shock.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/fluids/burgers_shock/equation.py
r"""Physics → tensor translation.

The viscous Burgers equation is the simplest model of a fluid that carries itself along:

    u_t + u u_x = ν u_xx      ⇔      u_t = -(½ u^2)_x + ν u_xx

On a periodic grid x_i the derivatives become matrices: (∂_x f)_i = D_{ij} f_j and
(∂_x^2 f)_i = L_{ij} f_j, so the PDE turns into a system of ODEs (method of lines)

    \dot u_i = -½ D_{ij} (u_j)^2 + ν L_{ij} u_j

The repeated index j is summed: each is one einsum('ij,j->i'). The square (u_j)^2 is elementwise.
"""

import torch


def rhs(u: torch.Tensor, D: torch.Tensor, L: torch.Tensor, nu: float) -> torch.Tensor:
    r"""\dot u_i = -½ D_{ij} u_j^2 + ν L_{ij} u_j   (j summed).   u: (N,), D, L: (N, N) → (N,)"""
    flux = 0.5 * u**2  # (N,) the "momentum flux" ½u², squared elementwise
    return -torch.einsum("ij,j->i", D, flux) + nu * torch.einsum("ij,j->i", L, u)


def slope(u: torch.Tensor, D: torch.Tensor) -> torch.Tensor:
    r"""u_x as (∂_x u)_i = D_{ij} u_j for one or many snapshots.   u: (..., N) → (..., N)"""
    return torch.einsum("ij,...j->...i", D, u)


def mass(u: torch.Tensor, dx: float) -> torch.Tensor:
    r"""∫ u dx ≈ Σ_i u_i Δx — conserved on a periodic domain because the PDE is a pure divergence."""
    return u.sum(dim=-1) * dx


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/fluids/burgers_shock/tensors.py
"""Build the concrete tensors: the periodic grid, the derivative matrices and the initial wave."""

import math

import torch

LENGTH = 2 * math.pi  # periodic domain [0, 2π)


def grid(n: int, device: torch.device) -> tuple[torch.Tensor, float]:
    """N equally spaced points x_i on [0, 2π) and the spacing Δx."""
    dx = LENGTH / n
    return torch.arange(n, device=device, dtype=torch.float64) * dx, dx


def derivative_matrices(n: int, dx: float, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Periodic central differences as (N, N) matrices.

    D: (f_{i+1} - f_{i-1}) / (2Δx)       first derivative
    L: (f_{i+1} - 2 f_i + f_{i-1}) / Δx²  second derivative
    The index i ± 1 wraps around (torch.roll of the identity matrix).
    """
    eye = torch.eye(n, device=device, dtype=torch.float64)
    right = torch.roll(eye, 1, dims=1)  # right[i, i+1] = 1
    left = torch.roll(eye, -1, dims=1)  # left[i, i-1] = 1
    D = (right - left) / (2 * dx)
    L = (right - 2 * eye + left) / dx**2
    return D, L


def initial_wave(x: torch.Tensor, amplitude: float, offset: float) -> torch.Tensor:
    """u_0(x) = U_0 + A sin x: shape (N,). The steepest downhill slope is -A, so a shock forms at t* = 1/A."""
    return offset + amplitude * torch.sin(x)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/fluids/burgers_shock/solve.py
"""March the method-of-lines ODEs with RK4, and evaluate the exact Cole–Hopf solution for comparison."""

import math

import torch

from problems._common.integrators import integrate

from .equation import rhs


def time_step(dx: float, u_max: float, nu: float) -> float:
    """Stable RK4 step: limited by advection (Δx/|u|) and by diffusion (Δx²/ν)."""
    return min(0.5 * dx / u_max, 0.25 * dx**2 / nu)


def simulate(
    u0: torch.Tensor, D: torch.Tensor, L: torch.Tensor, nu: float, dt: float, t_max: float
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), u (T, N))."""
    n_steps = int(math.ceil(t_max / dt))
    u = integrate(lambda u, t: rhs(u, D, L, nu), u0, dt, n_steps)
    times = torch.arange(n_steps + 1, device=u0.device, dtype=u0.dtype) * dt
    return times, u


def cole_hopf(x: torch.Tensor, t: float, nu: float, amplitude: float, offset: float, n_y: int = 6000) -> torch.Tensor:
    r"""Exact solution of viscous Burgers for u_0 = U_0 + A sin x (Cole–Hopf transform).

    u = -2ν ∂_x ln φ turns Burgers into the heat equation φ_t = ν φ_xx, whose solution is a Gaussian
    average. The result is a weighted mean over starting points y:

        u(x, t) = Σ_y w(x, y) (x - y) / t,     w = softmax_y[ -(x - y)² / (4νt) - F(y) / (2ν) ]

    with F(y) = ∫_0^y u_0 = U_0 y + A (1 - cos y). softmax keeps the huge exponents finite.
    """
    if t <= 0:
        return offset + amplitude * torch.sin(x)
    y = torch.linspace(-3 * math.pi, 5 * math.pi, n_y, dtype=torch.float64, device=x.device)  # (M,)
    F = offset * y + amplitude * (1 - torch.cos(y))  # (M,)
    gap = x[:, None] - y[None, :]  # (N, M)
    w = torch.softmax(-(gap**2) / (4 * nu * t) - F / (2 * nu), dim=1)  # (N, M), rows sum to 1
    return torch.einsum("nm,nm->n", w, gap) / t


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/fluids/burgers_shock/plot.py
"""Top: u(x) steepening into a shock, with the exact Cole–Hopf solution. Bottom: steepest slope vs time."""

from pathlib import Path
from typing import Callable

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(
    x: torch.Tensor, times: torch.Tensor, u: torch.Tensor, exact: Callable[[float], torch.Tensor],
    steepness: torch.Tensor, t_star: float, path: Path,
) -> Path:  # fmt: skip
    """exact(t) returns the Cole–Hopf solution at time t; it is evaluated only for the GIF frames."""
    xs, t, us, steep = x.cpu().numpy(), times.cpu().numpy(), u.cpu().numpy(), steepness.cpu().numpy()
    frames = frame_indices(len(t), 90)
    reference = {k: exact(float(t[k])).cpu().numpy() for k in frames}

    fig, (ax, ax2) = plt.subplots(2, 1, figsize=(7, 4.6), gridspec_kw={"height_ratios": [1.6, 1]})
    ax.plot(xs, us[0], color="#9ca3af", lw=1, label="u₀")
    line = ax.plot(xs, us[0], color="#2563eb", lw=2, label="finite differences")[0]
    ref = ax.plot(xs, us[0], color="#111827", lw=1, ls="--", label="Cole–Hopf (exact)")[0]
    pad = 0.15 * (us.max() - us.min())
    ax.set_xlim(0, xs[-1])
    ax.set_ylim(us.min() - pad, us.max() + pad)
    ax.set_xlabel("x")
    ax.set_ylabel("u")
    ax.legend(loc="lower left", fontsize=8)
    title = ax.set_title(" ")

    ax2.set_xlim(0, t[-1])
    ax2.set_ylim(0, steep.max() * 1.1)
    ax2.axvline(t_star, color="#f59e0b", ls="--", lw=1.2)
    ax2.text(t_star, steep.max() * 0.08, " t* = 1/max(−u₀′)", color="#b45309", fontsize=8, va="bottom")
    ax2.set_xlabel("t")
    ax2.set_ylabel("max(−u_x)")
    curve = ax2.plot([], [], color="#dc2626", lw=1.8)[0]

    def update(step: int):
        line.set_ydata(us[step])
        ref.set_ydata(reference[step])
        curve.set_data(t[: step + 1], steep[: step + 1])
        title.set_text(f"viscous Burgers   t = {t[step]:.2f}")

    fig.tight_layout()
    return save_gif(fig, update, frames, path, fps=15)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'nu': 0.02,
    'amplitude': 1.0,
    'offset': 0.0,
    't_max': 3.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.fluids.burgers_shock")]:
    del sys.modules[name]
main = importlib.import_module("problems.fluids.burgers_shock.main")
problem_dir = Path("problems/fluids/burgers_shock")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
